# 02 · Train LoRA on Nemotron-3-Nano-30B (Kaggle GPU)

Upload this notebook to Kaggle, attach the competition dataset and the
`metric/nemotron-3-nano-30b-a3b-bf16` model, then run top to bottom.

Output: `/kaggle/working/submission.zip` containing the LoRA adapter.

## 0. Environment

Kaggle's notebook images already include PyTorch + Transformers. We install
PEFT, accelerate, TRL, and bits/pieces the demo needs.

In [ ]:
%pip install -q -U peft accelerate trl datasets

## 1. Load training data

In [ ]:
import polars as pl

train = pl.read_csv('/kaggle/input/nvidia-nemotron-3-reasoning-challenge/train.csv')
print(train.shape)
train.head(3)

## 2. Format SFT rows (prompt + boxed answer)

In [ ]:
SYSTEM_PROMPT = (
    "You are solving an 'Alice's Wonderland' reasoning puzzle. "
    "Read the input/output examples to infer the hidden rule, then apply the rule "
    "to the final input. Reason step by step. Place your final answer inside "
    "\\boxed{...}. Do not output anything after the closing brace."
)

def format_row(prompt: str, answer: str) -> dict:
    return {
        'messages': [
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': prompt},
            {'role': 'assistant', 'content': f'\\boxed{{{answer}}}'},
        ]
    }

rows = [format_row(p, a) for p, a in zip(train['prompt'].to_list(), train['answer'].to_list())]
print('SFT rows:', len(rows))
rows[0]

## 3. Load Nemotron-3-Nano-30B + attach LoRA

In [ ]:
import site
# The competition's reference notebook includes a CUTLASS DSL helper required by the model.
# Adjust the path if you copy this notebook outside the official Kaggle competition environment.
cutlass_pkg_path = '/kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script/nvidia_cutlass_dsl/python_packages/'
site.addsitedir(cutlass_pkg_path)

import kagglehub, torch
from peft import LoraConfig, get_peft_model, TaskType
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_PATH = kagglehub.model_download('metric/nemotron-3-nano-30b-a3b-bf16/transformers/default')
OUTPUT_DIR = '/kaggle/working'
LORA_RANK = 32  # grader enforces max_lora_rank=32

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    device_map='auto',
    trust_remote_code=True,
    dtype=torch.bfloat16,
)

lora_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=16,
    target_modules=r'.*\\.(in_proj|out_proj|up_proj|down_proj)$',
    lora_dropout=0.05,
    bias='none',
    task_type=TaskType.CAUSAL_LM,
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 4. Train (your code here)

Drop your `SFTTrainer` / custom training loop in this cell. A minimal sketch:

```python
from datasets import Dataset
from trl import SFTTrainer, SFTConfig

ds = Dataset.from_list(rows)
cfg = SFTConfig(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    num_train_epochs=1,
    learning_rate=2e-4,
    bf16=True,
    logging_steps=10,
    save_strategy='no',
)
trainer = SFTTrainer(model=model, args=cfg, train_dataset=ds, tokenizer=tokenizer)
trainer.train()
```

In [ ]:
# TODO: replace this with your real training loop.
# For a smoke test, the demo just saves the un-trained adapter and zips it.
pass

## 5. Save adapter and package submission.zip

In [ ]:
model.save_pretrained(OUTPUT_DIR)
import os, subprocess
os.chdir(OUTPUT_DIR)
subprocess.run('zip -m submission.zip adapter_config.json adapter_model.safetensors', shell=True, check=True)
print('Wrote /kaggle/working/submission.zip')